<a href="https://colab.research.google.com/github/praveenkumarrlgmp-source/meachin-learing-project/blob/main/Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==========================================
# 1. LOAD DATASET
# ==========================================

df = pd.read_csv("/content/car_price_prediction_.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())


# ==========================================
# 2. REMOVE UNNECESSARY COLUMNS
# ==========================================

# Remove common ID/index columns if they exist
remove_columns = []

for col in ["ID", "id", "Car_ID", "Unnamed: 0"]:
    if col in df.columns:
        remove_columns.append(col)

if remove_columns:
    df = df.drop(remove_columns, axis=1)


# ==========================================
# 3. CREATE CAR AGE IF YEAR EXISTS
# ==========================================

if "Year" in df.columns:
    df["Car_Age"] = 2026 - df["Year"]

    # Year may still contain useful information,
    # so we keep it.


# ==========================================
# 4. FEATURES AND TARGET
# ==========================================

X = df.drop("Price", axis=1)
y = df["Price"]


# ==========================================
# 5. IDENTIFY DATA TYPES
# ==========================================

categorical_columns = X.select_dtypes(
    include=["object"]
).columns

numerical_columns = X.select_dtypes(
    exclude=["object"]
).columns


print("\nCategorical Columns:")
print(list(categorical_columns))

print("\nNumerical Columns:")
print(list(numerical_columns))


# ==========================================
# 6. PREPROCESSING
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_columns
        )
    ],
    remainder="passthrough"
)


# ==========================================
# 7. RANDOM FOREST PIPELINE
# ==========================================

pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "random_forest",
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        )
    )
])


# ==========================================
# 8. TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ==========================================
# 9. HYPERPARAMETER TUNING
# ==========================================

param_grid = {
    "random_forest__n_estimators": [
        200,
        300,
        500
    ],

    "random_forest__max_depth": [
        None,
        10,
        15,
        20,
        30
    ],

    "random_forest__min_samples_split": [
        2,
        5,
        10
    ],

    "random_forest__min_samples_leaf": [
        1,
        2,
        4
    ],

    "random_forest__max_features": [
        0.7,
        0.8,
        1.0
    ]
}


search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=20,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)


# ==========================================
# 10. TRAIN
# ==========================================

search.fit(
    X_train,
    y_train
)


# ==========================================
# 11. BEST MODEL
# ==========================================

model = search.best_estimator_

print("\n======================================")
print("BEST PARAMETERS")
print("======================================")

print(search.best_params__)


# ==========================================
# 12. PREDICTION
# ==========================================

y_pred = model.predict(X_test)


# ==========================================
# 13. EVALUATION
# ==========================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse ** 0.5

r2 = r2_score(
    y_test,
    y_pred
)


# ==========================================
# 14. PRINT RESULTS
# ==========================================

print("\n======================================")
print("RANDOM FOREST REGRESSION RESULTS")
print("======================================")

print(f"MAE      : {mae:.4f}")
print(f"MSE      : {mse:.4f}")
print(f"RMSE     : {rmse:.4f}")
print(f"R2 Score : {r2:.4f}")

print(f"\nR2 Percentage : {r2 * 100:.2f}%")


#

Dataset Shape: (2500, 10)

Columns:
['Car ID', 'Brand', 'Year', 'Engine Size', 'Fuel Type', 'Transmission', 'Mileage', 'Condition', 'Price', 'Model']

First 5 Rows:
   Car ID  Brand  Year  Engine Size Fuel Type Transmission  Mileage Condition  \
0       1  Tesla  2016          2.3    Petrol       Manual   114832       New   
1       2    BMW  2018          4.4  Electric       Manual   143190      Used   
2       3   Audi  2013          4.5  Electric       Manual   181601       New   
3       4  Tesla  2011          4.1    Diesel    Automatic    68682       New   
4       5   Ford  2009          2.6    Diesel       Manual   223009  Like New   

      Price     Model  
0  26613.92   Model X  
1  14679.61  5 Series  
2  44402.61        A4  
3  86374.33   Model Y  
4  73577.10   Mustang  

Categorical Columns:
['Brand', 'Fuel Type', 'Transmission', 'Condition', 'Model']

Numerical Columns:
['Car ID', 'Year', 'Engine Size', 'Mileage', 'Car_Age']
Fitting 5 folds for each of 20 candidates, to

KeyboardInterrupt: 